# EnergyPredict — Renewable Forecasting + Exploratory Net-Load Analysis

**Part A — Forecast renewable output.**  
This notebook forecasts combined solar + wind generation 24 hours ahead using the same four non-LSTM models used for the Paper 2 comparison:
1. Persistence baseline
2. Ridge Regression
3. Random Forest
4. Gradient Boosting

The LSTM used in the demand study is intentionally not part of the renewable comparison because it was not tuned for a second target.

**Part B — Exploratory net-load analysis.**  
We combine the demand and renewable forecasts to calculate a predicted gap. This section is exploratory and is **not** presented as a validated dispatch optimization or operational recommendation in the paper.

## Load the data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("energy_data_final.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

print("Rows:", len(df))
df[["timestamp", "demand_MW", "solar_MW", "wind_MW", "renewable_MW"]].head()


Rows: 8544


,timestamp,demand_MW,solar_MW,wind_MW,renewable_MW
0,2025-01-08 00:00:00+00:00,24153.0,9913.0,3571.0,13484.0
1,2025-01-08 01:00:00+00:00,23668.0,6674.0,3456.0,10130.0
2,2025-01-08 02:00:00+00:00,24645.0,1134.0,3373.0,4507.0
3,2025-01-08 03:00:00+00:00,26672.0,-18.0,3766.0,3748.0
4,2025-01-08 04:00:00+00:00,26817.0,-53.0,4370.0,4317.0


## Part A: Forecasting renewable output

## Build features specifically for renewables

The demand model used lag features built from `demand_MW`. For renewables, we need the *same idea* but built from `renewable_MW` instead. solar and wind don't follow the same rhythm as human electricity usage, so they need their own lag columns.

In [2]:
df["renewable_lag_24h"] = df["renewable_MW"].shift(24)
df["renewable_lag_48h"] = df["renewable_MW"].shift(48)
df["renewable_lag_168h"] = df["renewable_MW"].shift(168)
df["renewable_rolling_mean_24h"] = df["renewable_MW"].shift(1).rolling(24).mean()

# What we're trying to predict: renewable output 24 hours from now
df["target_renewable_24h"] = df["renewable_MW"].shift(-24)

df.filter(like="renewable").head()


,renewable_MW,renewable_lag_24h,renewable_lag_48h,renewable_lag_168h,renewable_rolling_mean_24h,target_renewable_24h
0,13484.0,NaN,NaN,NaN,NaN,14211.0
1,10130.0,NaN,NaN,NaN,NaN,10617.0
2,4507.0,NaN,NaN,NaN,NaN,3023.0
3,3748.0,NaN,NaN,NaN,NaN,1257.0
4,4317.0,NaN,NaN,NaN,NaN,1130.0


In [3]:
renewable_feature_columns = [
    "renewable_MW", "renewable_lag_24h", "renewable_lag_48h", "renewable_lag_168h",
    "renewable_rolling_mean_24h",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
    "is_weekend", "is_holiday",
    "temperature_C", "humidity_pct", "dewpoint_C", "wind_speed_kmh",
]

model_df = df.dropna(subset=renewable_feature_columns + ["target_renewable_24h"]).reset_index(drop=True)
print("Rows ready for modeling:", len(model_df))


Rows ready for modeling: 8352


## Split into train and test (chronological rule as before)

In [4]:
cutoff_date = model_df["timestamp"].max() - pd.Timedelta(days=60)

train = model_df[model_df["timestamp"] < cutoff_date].reset_index(drop=True)
test = model_df[model_df["timestamp"] >= cutoff_date].reset_index(drop=True)

X_train = train[renewable_feature_columns]
y_train = train["target_renewable_24h"]
X_test = test[renewable_feature_columns]
y_test = test["target_renewable_24h"]

print("Training rows:", len(train))
print("Testing rows:", len(test))


Training rows: 6911
Testing rows: 1441


## Score function

Same idea as before but solar output is close to zero every night, so calculating a *percentage* error at those hours would involve dividing by (almost) zero and give a meaningless number. We only calculate MAPE on hours where output is meaningfully above zero.

In [5]:
def score_model(actual_values, predicted_values):
    predicted_values = np.clip(predicted_values, 0, None)
    errors = actual_values - predicted_values
    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))
    meaningful_hours = np.abs(actual_values) > 200
    mape = np.mean(np.abs(errors[meaningful_hours] / actual_values[meaningful_hours])) * 100
    return mae, rmse, mape

renewable_results = []

## Baseline — "in 24 hours, output will look like right now"

In [6]:
baseline_prediction = test["renewable_MW"].values
baseline_mae, baseline_rmse, baseline_mape = score_model(y_test.values, baseline_prediction)
renewable_results.append(("Persistence (baseline)", baseline_mae, baseline_rmse, baseline_mape))
print("Baseline MAE:", round(baseline_mae, 1), "MW")
print("Baseline RMSE:", round(baseline_rmse, 1), "MW")
print("Baseline MAPE:", round(baseline_mape, 2), "%")

Baseline MAE: 1153.8 MW
Baseline RMSE: 1784.7 MW
Baseline MAPE: 45.88 %


## Ridge, Random Forest, and Gradient Boosting

In [7]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

ridge_model = Ridge(alpha=10.0)
ridge_model.fit(X_train, y_train)
ridge_pred = ridge_model.predict(X_test)
ridge_mae, ridge_rmse, ridge_mape = score_model(y_test.values, ridge_pred)
renewable_results.append(("Ridge Regression", ridge_mae, ridge_rmse, ridge_mape))

forest_model = RandomForestRegressor(
    n_estimators=300, max_depth=14, random_state=42, n_jobs=-1
)
forest_model.fit(X_train, y_train)
forest_pred = forest_model.predict(X_test)
forest_mae, forest_rmse, forest_mape = score_model(y_test.values, forest_pred)
renewable_results.append(("Random Forest", forest_mae, forest_rmse, forest_mape))

boosting_model = GradientBoostingRegressor(
    n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42
)
boosting_model.fit(X_train, y_train)
boosting_pred = boosting_model.predict(X_test)
boosting_mae, boosting_rmse, boosting_mape = score_model(y_test.values, boosting_pred)
renewable_results.append(("Gradient Boosting", boosting_mae, boosting_rmse, boosting_mape))

print("Ridge:", round(ridge_mae,1), round(ridge_rmse,1), round(ridge_mape,2))
print("Random Forest:", round(forest_mae,1), round(forest_rmse,1), round(forest_mape,2))
print("Gradient Boosting:", round(boosting_mae,1), round(boosting_rmse,1), round(boosting_mape,2))

Ridge: 1098.4 1691.7 53.16
Random Forest: 1295.3 1886.0 80.04
Gradient Boosting: 1282.0 1990.5 70.1


## LSTM

Same structure as the demand notebook's LSTM. Sequences of 168 hours, scaled, trained with early stopping. The only difference is *what* it's trying to predict.

## LSTM omitted from the primary renewable comparison

The demand study included an LSTM, but Paper 2 intentionally excludes it from the renewable comparison.
The reason is methodological: the LSTM was not tuned for the renewable target, and including an untuned deep-learning model would make the comparison less fair.

The four-model renewable comparison therefore uses the same core non-LSTM models as the paper:
Persistence, Ridge Regression, Random Forest, and Gradient Boosting.

In [8]:
# LSTM code intentionally disabled for Paper 2 primary results.

In [9]:
# No LSTM model is trained in the Paper 2 primary experiment.
# This cell is intentionally retained as a placeholder so notebook section numbering stays stable.

## Compare the four renewable models used in Paper 2

In [10]:
results_table = pd.DataFrame(
    renewable_results,
    columns=["Model", "MAE (MW)", "RMSE (MW)", "MAPE (%)"]
)

baseline_mae_value = results_table.loc[
    results_table["Model"] == "Persistence (baseline)", "MAE (MW)"
].iloc[0]

results_table["Improvement vs baseline"] = (
    (baseline_mae_value - results_table["MAE (MW)"]) /
    baseline_mae_value * 100
).round(1)

results_table

,Model,MAE (MW),RMSE (MW),MAPE (%),Improvement vs baseline
0,Persistence (baseline),1153.815406,1784.730485,45.875121,0.0
1,Ridge Regression,1098.440773,1691.661137,53.158979,4.8
2,Random Forest,1295.270589,1886.046682,80.044649,-12.3
3,Gradient Boosting,1281.998001,1990.473710,70.104266,-11.1


### Interpretation

For demand, the tree-based models clearly beat persistence. For renewables, the result is different:
the simpler Ridge model is the best of the four models by MAE.

The key lesson is not that complex models are bad. It is that model complexity should match the
amount of learnable structure in the target.

In [11]:
# Figure 1 — Renewable model performance
# This figure corresponds to Table 2 in the paper and uses the same three metrics.

import os
os.makedirs("cigre_figures", exist_ok=True)

plot_table = results_table.copy()
x = np.arange(len(plot_table))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, column, title in zip(
    axes,
    ["MAE (MW)", "RMSE (MW)", "MAPE (%)"],
    ["Mean Absolute Error", "Root Mean Squared Error", "Mean Absolute Percentage Error"]
):
    values = plot_table[column].values
    ax.bar(x, values)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_table["Model"], rotation=25, ha="right")
    ax.set_ylabel(column)
    ax.grid(axis="y", alpha=0.25)
    for xi, value in zip(x, values):
        label = f"{value:,.1f}" if "MW" in column else f"{value:.2f}"
        ax.text(xi, value, label, ha="center", va="bottom", fontsize=9)

fig.suptitle("Figure 1: Renewable Forecasting Model Performance", fontsize=14)
fig.tight_layout()
fig.savefig("cigre_figures/Figure_1_Renewable_Model_Performance.png",
            dpi=300, bbox_inches="tight")
plt.show()

/var/folders/kc/2hr2jfpx4_z05kd0_c4dxvkr0000gn/T/ipykernel_39878/3965207254.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
best_row = results_table.loc[results_table["MAE (MW)"].idxmin()]
print("Best renewable model:", best_row["Model"],
      "with MAE", round(best_row["MAE (MW)"], 1), "MW")

# Paper 2 identifies Ridge as the best model by MAE.
final_renewable_prediction = np.clip(ridge_pred, 0, None)

Best renewable model: Ridge Regression with MAE 1098.4 MW


## Part B: Turning two forecasts into a recommendation

## Quickly retrain the demand model (so this notebook can run on its own)

This repeats the Gradient Boosting steps from the demand notebook, just condensed, so we have a demand prediction to pair with our renewable prediction.

In [13]:
demand_columns_to_exclude = ["timestamp", "target_t_plus_24h", "target_t_plus_48h"] + [
    "renewable_lag_24h", "renewable_lag_48h", "renewable_lag_168h",
    "renewable_rolling_mean_24h", "target_renewable_24h",
]
demand_feature_columns = [c for c in model_df.columns if c not in demand_columns_to_exclude]

demand_model = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42)
demand_model.fit(train[demand_feature_columns], train["target_t_plus_24h"])
final_demand_prediction = demand_model.predict(test[demand_feature_columns])

print("Demand model ready.")


Demand model ready.


## Combine the two forecasts

For every hour in the test period, we now have:
- **Predicted demand** 24 hours from now
- **Predicted renewable output** 24 hours from now

The difference between them (`gap`) tells us how much power will need to come from *non-renewable* sources like natural gas.

In [14]:
dispatch = pd.DataFrame({
    "timestamp": test["timestamp"].values,
    "predicted_demand": final_demand_prediction,
    "predicted_renewable": final_renewable_prediction,
})
dispatch["gap"] = dispatch["predicted_demand"] - dispatch["predicted_renewable"]

dispatch.head()


,timestamp,predicted_demand,predicted_renewable,gap
0,2025-10-29 23:00:00,28506.567229,16571.840642,11934.726586
1,2025-10-30 00:00:00,28882.804341,15754.847045,13127.957295
2,2025-10-30 01:00:00,28360.779893,12052.597873,16308.182020
3,2025-10-30 02:00:00,29246.914535,3877.481124,25369.433411
4,2025-10-30 03:00:00,29475.540556,1465.289014,28010.251543


## Set the rule for flagging hours

- **High stress**: the gap is in the top 10% of all gaps we see in the test period — these are the hours needing the most backup power
- **Renewable surplus**: predicted renewables cover at least half of predicted demand — a good time to store extra energy
- Otherwise: **Normal**

In [15]:
# Apply the rule described above to actually label every test hour.
stress_cutoff = dispatch["gap"].quantile(0.90)

dispatch["status"] = "Normal"
dispatch.loc[dispatch["gap"] >= stress_cutoff, "status"] = "High stress"
dispatch.loc[dispatch["predicted_renewable"] >= 0.5 * dispatch["predicted_demand"], "status"] = "Renewable surplus"

print("Stress cutoff (90th percentile of gap):", round(stress_cutoff), "MW")
print()
print(dispatch["status"].value_counts())


Stress cutoff (90th percentile of gap): 24528 MW

status
Normal               1177
High stress           145
Renewable surplus     119
Name: count, dtype: int64


## Figure 2 — Gradient Boosting feature importance

This reproduces the feature-importance analysis reported in Section 4.4 of the paper.

In [16]:
raw_importance = pd.Series(
    boosting_model.feature_importances_,
    index=renewable_feature_columns
)

importance_values = pd.Series({
    "Current renewable output": raw_importance.get("renewable_MW", 0),
    "Renewable lag 48h": raw_importance.get("renewable_lag_48h", 0),
    "Renewable lag 168h": raw_importance.get("renewable_lag_168h", 0),
    "Temperature": raw_importance.get("temperature_C", 0),
    "All other features": max(
        0,
        1 - (
            raw_importance.get("renewable_MW", 0)
            + raw_importance.get("renewable_lag_48h", 0)
            + raw_importance.get("renewable_lag_168h", 0)
            + raw_importance.get("temperature_C", 0)
        )
    )
}).sort_values()

plt.figure(figsize=(9, 5))
plt.barh(importance_values.index, importance_values.values * 100)
plt.title("Figure 2: Gradient Boosting Feature Importance")
plt.xlabel("Feature importance [%]")
plt.grid(axis="x", alpha=0.25)

for y, value in enumerate(importance_values.values * 100):
    plt.text(value + 0.3, y, f"{value:.1f}%", va="center")

plt.tight_layout()
plt.savefig("cigre_figures/Figure_2_Renewable_Feature_Importance.png",
            dpi=300, bbox_inches="tight")
plt.show()

/var/folders/kc/2hr2jfpx4_z05kd0_c4dxvkr0000gn/T/ipykernel_39878/2348468329.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Exploratory dispatch analysis — not a validated operational model

The following analysis combines predicted demand and predicted renewable output to calculate an exploratory
net-load gap. The paper does not claim that these flags are an operational dispatch optimization.

In [17]:
dispatch["hour"] = (
    pd.to_datetime(dispatch["timestamp"], utc=True)
      .dt.tz_convert("America/Los_Angeles")
      .dt.hour
)

stress_by_hour = dispatch[dispatch["status"] == "High stress"]["hour"].value_counts().sort_index()
surplus_by_hour = dispatch[dispatch["status"] == "Renewable surplus"]["hour"].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].bar(stress_by_hour.index, stress_by_hour.values)
axes[0].set_title("Exploratory High-Stress Hours by Time of Day")
axes[0].set_xlabel("Hour (local time)")
axes[0].set_ylabel("Number of hours")
axes[0].grid(axis="y", alpha=0.25)

axes[1].bar(surplus_by_hour.index, surplus_by_hour.values)
axes[1].set_title("Exploratory Renewable-Surplus Hours by Time of Day")
axes[1].set_xlabel("Hour (local time)")
axes[1].set_ylabel("Number of hours")
axes[1].grid(axis="y", alpha=0.25)

fig.suptitle("Figure 3: Exploratory Net-Load Signals", fontsize=14)
fig.tight_layout()
fig.savefig("cigre_figures/Figure_3_Exploratory_Net_Load_Signals.png",
            dpi=300, bbox_inches="tight")
plt.show()

/var/folders/kc/2hr2jfpx4_z05kd0_c4dxvkr0000gn/T/ipykernel_39878/3147549460.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
# Figure 4 — One-week sample of predicted demand and renewable supply
sample_week = dispatch[
    (dispatch["timestamp"] >= "2025-11-10") &
    (dispatch["timestamp"] < "2025-11-17")
]

plt.figure(figsize=(12, 5))
plt.plot(sample_week["timestamp"], sample_week["predicted_demand"],
         label="Predicted demand", linewidth=1.5)
plt.plot(sample_week["timestamp"], sample_week["predicted_renewable"],
         label="Predicted renewable", linewidth=1.5)

stress_points = sample_week[sample_week["status"] == "High stress"]
plt.scatter(
    stress_points["timestamp"],
    stress_points["predicted_demand"],
    zorder=3,
    label="Exploratory high-stress flag"
)

plt.title("Figure 4: Sample Week — Predicted Demand vs. Renewable Supply")
plt.xlabel("Date")
plt.ylabel("Power [MW]")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig("cigre_figures/Figure_4_Sample_Week_Forecast.png",
            dpi=300, bbox_inches="tight")
plt.show()

/var/folders/kc/2hr2jfpx4_z05kd0_c4dxvkr0000gn/T/ipykernel_39878/2867291821.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Optional backtest check of the exploratory high-stress rule

This is an exploratory diagnostic only. It should not be presented as a validated operational reliability metric.

In [19]:
dispatch["actual_demand"] = test["target_t_plus_24h"].values
dispatch["actual_renewable"] = y_test.values   # from Step 3

dispatch["actual_gap"] = dispatch["actual_demand"] - dispatch["actual_renewable"]
actual_stress_cutoff = dispatch["actual_gap"].quantile(0.90)

dispatch["really_was_high_stress"] = dispatch["actual_gap"] >= actual_stress_cutoff
dispatch["we_flagged_high_stress"] = dispatch["status"] == "High stress"

true_positives = (dispatch["we_flagged_high_stress"] & dispatch["really_was_high_stress"]).sum()
false_positives = (dispatch["we_flagged_high_stress"] & ~dispatch["really_was_high_stress"]).sum()
false_negatives = (~dispatch["we_flagged_high_stress"] & dispatch["really_was_high_stress"]).sum()

precision = true_positives / (true_positives + false_positives)
recall = true_positives / (true_positives + false_negatives)

print("Precision:", round(precision * 100, 1), "% - of hours we flagged, this % really were high-stress")
print("Recall:   ", round(recall * 100, 1), "% - of hours that really were high-stress, we caught this %")


Precision: 74.5 % - of hours we flagged, this % really were high-stress
Recall:    74.0 % - of hours that really were high-stress, we caught this %


## Conclusion

- The Paper 2 primary renewable experiment compares four models: persistence, Ridge Regression, Random Forest, and Gradient Boosting.
- Ridge is the best model by MAE in the reported renewable results.
- The notebook now reports MAE, RMSE, and MAPE consistently with the paper.
- Figures 1–2 correspond directly to the paper's renewable model results and feature-importance discussion.
- The dispatch/net-load section is retained only as an exploratory analysis; it is not claimed as a validated dispatch optimization.